
1. What is DevOps in Databricks?
DevOps combines development practices and operations practices to deliver data applications reliably.

In a Data Engineering project, it includes:
- Version-controlling notebooks, Python files, SQL and configuration in Git.
- Testing code before deployment.
- Packaging Databricks resources as deployable configuration.
- Deploying the same project to development, UAT and production.
- Monitoring failures, performance and costs.
- Controlling permissions, secrets and production changes.
Interview answer: I use Git for version control, automated tests in CI, Databricks Asset Bundles for packaging and deployment, and environment-specific configuration to promote the same data engineering project across development, UAT and production.
2. CI vs Continuous Delivery vs Continuous Deployment
Practice	Meaning	Example
Continuous Integration (CI)	Automatically test and validate code changes	Run PySpark unit tests when a pull request is created
Continuous Delivery	Keep changes deployable; production deployment requires approval	Deploy to UAT, then approve production
Continuous Deployment	Automatically deploy every change that passes required checks	Successful main-branch pipeline deploys to production

Remember:
- CI = integrate and test.
- Continuous Delivery = ready to deploy.
- Continuous Deployment = deploy automatically.
For a production data platform, continuous delivery with a production approval gate is often a sensible starting point. Automatically deploying every change to production is not automatically a good idea.
3. Git workflow for Databricks
A simple team workflow:
main
 |
 +---- feature/bronze-ingestion
 |           |
 |           +---- developer changes
 |           +---- unit tests
 |           +---- pull request
 |
 +---- review and merge
             |
             v
         CI pipeline
             |
             v
         Dev deployment
             |
             v
         UAT testing
             |
             v
       Production approval
             |
             v
       Production deployment


Recommended Git structure:
E-Commerce-Lakehouse/
│
├── notebooks/
│   ├── bronze_ingestion.py
│   ├── silver_transformation.py
│   └── gold_aggregation.py
│
├── src/
│   ├── ingestion/
│   ├── transformations/
│   └── quality/
│
├── tests/
│   ├── test_ingestion.py
│   ├── test_transformations.py
│   └── test_quality.py
│
├── resources/
│   └── ecommerce_job.yml
│
├── databricks.yml
├── requirements.txt
└── README.md


Do not commit passwords, access tokens, private keys, .env files or other credentials to Git. Use Databricks secrets or an approved identity-based authentication method.
4. Databricks CLI
The Databricks CLI lets you manage workspace resources from your local terminal or automation scripts. It wraps Databricks REST APIs. 
Databricks on AWS

4.1 Install on Windows
Since you use Windows and PowerShell, open PowerShell or Command Prompt:
winget search databricks

winget install Databricks.DatabricksCLI


Close and reopen the terminal, then verify:
databricks -v


Use the current CLI, not the old legacy Python CLI version 0.18. 
Databricks on AWS

4.2 Authenticate with your Databricks workspace
Run:
databricks auth login --host https://YOUR-WORKSPACE-URL


Replace the URL with your actual workspace URL. Complete the browser login and save a profile, for example DEFAULT.

Check profiles:
databricks auth profiles


Check authentication:
databricks auth env --profile DEFAULT


For local development, interactive OAuth login is appropriate. For automated CI/CD, use a service principal with OAuth machine-to-machine authentication or workload identity federation rather than a developer's personal login. 
Databricks on AWS+1

4.3 Important CLI commands
Run these after authentication, using PowerShell or a terminal.
Category	Command	Purpose
Help	databricks -h	List command groups
Files	databricks fs ls dbfs:/	List DBFS files where enabled
Volumes	databricks fs ls dbfs:/Volumes/workspace/default/data_volume/	Inspect a volume path using the CLI filesystem interface
Workspace	databricks workspace ls /Workspace/Users/	List workspace objects
Catalogs	databricks catalogs list	List catalogs
Schemas	databricks schemas list workspace	List schemas in a catalog
Tables	databricks tables list workspace ecommerce_gold	List tables in a schema
Clusters	databricks clusters list	List clusters
Jobs	databricks jobs list	List jobs
Secrets	databricks secrets list-scopes	List secret scope names

Your environment has DBFS limitations. Prefer Unity Catalog Volumes and workspace files; do not assume dbfs:/FileStore is available. Also, the CLI's fs commands and the workspace's /Volumes/... path are different interfaces, so verify the path syntax supported by your installed CLI version.

Additional useful commands:
# Show a specific job
databricks jobs get JOB_ID

# Show a specific cluster
databricks clusters get CLUSTER_ID

# List workspace files
databricks workspace ls /Workspace

# List secret names in a scope
databricks secrets list ecommerce_scope


Security: Listing secret scopes or secret names is not the same as retrieving secret values. Never print credentials in terminal logs or CI output.
5. Databricks Asset Bundles (DAB)
Important interview topic
Databricks documentation now calls these Declarative Automation Bundles, formerly Databricks Asset Bundles. The familiar abbreviation DAB and CLI command family databricks bundle are still widely used. 
Databricks on AWS+1


A bundle packages your project source code and Databricks resource definitions into a deployable unit.

It can define:
- Lakeflow Jobs
- Lakeflow Declarative Pipelines
- Notebooks and Python files
- SQL files
- Dashboards and other supported Databricks resources
- Environment-specific configuration
5.1 Bundle project structure
E-Commerce-Lakehouse/
│
├── databricks.yml
│
├── resources/
│   ├── ecommerce_job.yml
│   └── ecommerce_pipeline.yml
│
├── src/
│   ├── bronze_ingestion.py
│   ├── silver_transformation.py
│   └── gold_aggregation.py
│
├── tests/
│   ├── test_ingestion.py
│   └── test_transformations.py
│
├── requirements.txt
│
├── README.md
│
└── .gitignore


File / folder	Purpose
databricks.yml	Main bundle configuration
resources/	Job and pipeline definitions
src/	Python and notebook source files
tests/	Unit and integration tests
requirements.txt	Python dependencies
.gitignore	Excludes local and sensitive files

5.2 What is databricks.yml?
It is the main configuration file at the root of a bundle. It defines the bundle name, variables, included resource files and deployment targets. 
Databricks on AWS


Example:
bundle:
  name: ecommerce_lakehouse

include:
  - resources/*.yml

variables:
  catalog:
    description: Unity Catalog name
    default: workspace

  schema:
    description: Gold schema name
    default: ecommerce_gold

targets:

  dev:
    mode: development
    default: true

    workspace:
      host: https://YOUR-DEV-WORKSPACE-URL

  uat:
    mode: production

    workspace:
      host: https://YOUR-UAT-WORKSPACE-URL

  prod:
    mode: production

    workspace:
      host: https://YOUR-PROD-WORKSPACE-URL


Replace the example workspace URLs with your actual environment URLs. You can use the same workspace with separate catalogs for a learning exercise, but real production isolation is better achieved with separate workspaces and controlled catalog access.
5.3 What are bundle variables?
Variables allow you to reuse configuration without hardcoding environment-specific values.

For example:
variables:
  catalog:
    default: workspace


Reference it in resource configuration:
catalog: ${var.catalog}


For the three environments, you could use:
Target	Catalog
Dev	ecommerce_dev
UAT	ecommerce_uat
Prod	ecommerce_prod

The important point is that the same code can be deployed with different configuration values.

Do not simply deploy the dev catalog to production because it works. Validate that the production target has the correct permissions, storage locations, identities and data sources.
5.4 Bundle lifecycle commands
Command	Purpose
databricks bundle init	Create a bundle from a template
databricks bundle validate	Validate configuration
databricks bundle deploy -t dev	Deploy to development
databricks bundle deploy -t uat	Deploy to UAT
databricks bundle deploy -t prod	Deploy to production
databricks bundle run -t dev ecommerce_job	Run a deployed job
databricks bundle summary -t dev	Show deployed bundle resources
databricks bundle destroy -t dev	Remove resources managed by the bundle

bundle destroy is destructive. Don't use it as a routine cleanup command in a shared or production environment.
6. Hands-on: Package a Lakeflow Job as a DAB and deploy to Dev
Your required practical exercise
We'll create a small bundle for an E-Commerce Bronze ingestion job.
Step 1 — Create the bundle
In PowerShell:
cd $HOME\Downloads

mkdir Ecommerce-DAB
cd Ecommerce-DAB

databricks bundle init


Select these options in the interactive template:
Prompt	Selection
Template	default-python
Project name	ecommerce_lakehouse
Include a job	Yes
Include ETL pipeline	No for this exercise
Include sample Python package	No
Use serverless	Yes
Default catalog	An existing catalog, such as workspace
Personal schema	Yes for development

The generated template includes databricks.yml, a job definition under resources/, source files under src/, and tests. The serverless option is useful for your current Databricks setup. 
Databricks on Google Cloud

Step 2 — Inspect the generated structure
ecommerce_lakehouse/
│
├── databricks.yml
├── resources/
│   └── sample_job.job.yml
├── src/
│   └── sample_notebook.ipynb
├── tests/
└── README.md


Rename the sample job and notebook to meaningful project names if desired. Keep the references in the YAML consistent with the renamed files.
Step 3 — Understand the job resource file
The generated job YAML defines the job resource, its tasks and its compute settings.

For example, a job task definition conceptually looks like this:
resources:
  jobs:
    ecommerce_bronze_job:
      name: ecommerce_bronze_job

      tasks:
        - task_key: bronze_ingestion

          notebook_task:
            notebook_path: ../src/sample_notebook.ipynb


This is a simplified illustration of the resource definition, not a complete serverless job configuration. Keep the compute/environment settings generated by the CLI template rather than replacing them with an arbitrary cluster configuration.
Step 4 — Update bundle variables
Open databricks.yml and check that the catalog and schema values are appropriate for your development environment.

For example:
variables:
  catalog:
    description: Target catalog
    default: workspace

  schema:
    description: Target schema
    default: default


You can use ${var.catalog} and ${var.schema} in your job or pipeline configuration where those resource fields support variable substitution.
Step 5 — Validate the bundle
From the project root:
databricks bundle validate -t dev


Expected outcome: the CLI prints a summary of the resolved bundle configuration if validation succeeds.

If it fails, fix the reported YAML, resource, path or authentication issue before deploying.
Step 6 — Deploy to Dev
databricks bundle deploy -t dev


Deployment command
This deploys the bundle's resources and source files to the development workspace.
Step 7 — Verify the deployment
databricks bundle summary -t dev


Then open Databricks:
1. Go to Workspace and locate the deployed bundle files.
2. Open Jobs & Pipelines.
3. Find the development job.
4. Inspect its tasks and configuration.
Step 8 — Run the job
Use the job key defined in your resource YAML:
databricks bundle run -t dev sample_job


If you renamed the job key to ecommerce_bronze_job, use:
databricks bundle run -t dev ecommerce_bronze_job


The CLI returns run information. Open the run URL and inspect the job output.
Complete command sequence
# Check CLI
databricks -v

# Authenticate
databricks auth login --host https://YOUR-WORKSPACE-URL

# Initialize
databricks bundle init

# Enter project directory
cd ecommerce_lakehouse

# Validate
databricks bundle validate -t dev

# Deploy
databricks bundle deploy -t dev

# Verify deployment
databricks bundle summary -t dev

# Run the job
databricks bundle run -t dev sample_job


The sequence follows Databricks' documented job deployment workflow. 
Databricks on Google Cloud+1

7. GitHub Actions — CI pipeline
GitHub Actions automates tests and validation when developers push code or create pull requests.
7.1 CI workflow
GitHub Actions CI pipeline
Developer pushes code

GitHub feature branch / pull request


Install Python dependencies

Run PySpark unit tests

Validate DAB configuration

All checks pass

Pull request can proceed to review


Merge approved changes

7.2 Create a GitHub Actions file
Inside your repository create:
.github/
└── workflows/
    └── ci.yml


Example CI configuration:

name: E-Commerce Lakehouse CI

on:
  pull_request:
    branches: [main]
  push:
    branches: [main]

jobs:
  test-and-validate:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install pytest pyspark

      - name: Run unit tests
        run: |
          pytest -q

      - name: Install Databricks CLI
        uses: databricks/setup-cli@main

      - name: Validate bundle
        run: |
          databricks bundle validate -t dev
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}



This example assumes you have configured an OAuth service principal and stored the corresponding credentials in GitHub Actions secrets. For production, prefer workload identity federation where your cloud and GitHub configuration support it, avoiding long-lived client secrets. Pin third-party GitHub Actions to reviewed versions or commit SHAs in a production workflow.

Important: This is a CI workflow. It tests code and validates the bundle. It does not yet deploy to production.
7.3 What happens when a test fails?
For example, a developer changes the sales calculation incorrectly.
- GitHub Actions starts.
- pytest runs.
- A unit test fails.
- The workflow reports failure.
- The pull request cannot pass the required status check.
- The developer fixes the code and pushes again.
This prevents known failures from reaching the next deployment stage.
7.4 CI/CD promotion workflow
A fuller pipeline might be:
Feature branch
      |
      v
Pull request
      |
      v
Unit tests + bundle validation
      |
      v
Code review and merge
      |
      v
Deploy to DEV
      |
      v
Integration tests in UAT
      |
      v
Approval gate
      |
      v
Deploy to PROD


Databricks recommends bundles for managing resources and deployments as code, and service-principal-based authentication for automated deployment. 
Databricks on AWS+2

8. Unit testing PySpark
Unit testing means testing individual functions independently before running the entire pipeline.

For your E-Commerce project, test:
- Null-value handling.
- Duplicate removal.
- Data type conversion.
- Invalid order amounts.
- Revenue calculations.
- Business rules.
8.1 Separate transformation logic from notebook execution
Create:

src/transformations.py
from pyspark.sql import functions as Fdef clean_orders(df):    return (        df        .filter(F.col("order_id").isNotNull())        .filter(F.col("total_amount") > 0)        .dropDuplicates(["order_id"])    )def calculate_revenue(df):    return (        df.groupBy("order_date")        .agg(            F.sum("total_amount").alias("total_revenue")        )    )



8.2 Create a unit test
Create:

tests/test_transformations.py
import pytestfrom pyspark.sql import SparkSessionfrom src.transformations import (    clean_orders,    calculate_revenue)@pytest.fixture(scope="session")def spark():    session = (        SparkSession.builder        .master("local[2]")        .appName("ECommerceUnitTests")        .getOrCreate()    )    yield session    session.stop()def test_clean_orders(spark):    data = [        (1, "2026-11-16", 100.0),        (1, "2026-11-16", 100.0),        (2, "2026-11-16", -50.0),        (None, "2026-11-16", 200.0)    ]    columns = [



Run:
pip install pytest pyspark

pytest -v


Expected results:
test_clean_orders PASSED
test_calculate_revenue PASSED

2 passed


The output above is the expected result, not a test execution performed on your machine.

Important: These are unit tests using local Spark. They don't prove that the Databricks job, Unity Catalog permissions, external data access or production pipeline will work. You still need integration tests in a Databricks environment.
9. Cost optimization in Databricks
Cost optimization means reducing unnecessary compute, storage and query expenses without sacrificing required performance or reliability.
9.1 Cluster sizing
Cluster sizing means selecting suitable compute resources based on workload requirements.
Workload	Starting approach
Small development notebook	Small compute or serverless
Small scheduled ETL	Small job compute or serverless
Medium transformations	Scale according to data volume and shuffle
Large joins and aggregations	Larger distributed compute if benchmarks justify it
SQL dashboards	Serverless SQL Warehouse or appropriately sized SQL Warehouse
Continuous streaming	Compute sized for arrival rate and processing latency

Don't memorize "small cluster = 2 workers" as a universal rule. Worker count and machine types depend on cloud provider, workload, available compute and service limits.
9.2 Important cost optimization techniques
Technique	Explanation	E-Commerce example
Autoscaling	Adjust workers based on workload	Increase compute during large ETL runs
Auto termination	Shut down idle interactive compute	Stop unused development clusters
Job compute	Use task-specific compute	Run nightly ingestion separately
Serverless	Avoid manually managing infrastructure	Run supported notebooks and SQL workloads
Photon	Accelerate supported queries	Improve Gold aggregation performance
Incremental processing	Process only new or changed data	Ingest new orders rather than all historical orders
Partition pruning	Read only relevant partitions	Query a selected date range
File compaction	Reduce excessive small files	Optimize frequently queried Delta tables
Broadcast joins	Avoid unnecessary large shuffles	Join a small product dimension to large order data
Right-sized warehouse	Avoid oversized SQL compute	Match warehouse size to dashboard concurrency

9.3 Example: Full load vs incremental load
Full load

Every execution reads all historical orders.
Day 1: 1 million rows
    Day 2: 1 million rows
    Day 3: 1 million rows

Incremental load

Every execution processes only the newly arrived or changed records.
Day 1: 100,000 new rows
    Day 2: 20,000 new rows
    Day 3: 15,000 new rows
Illustrative example. Actual savings depend on source changes, query design, storage layout and execution overhead.
For your project, Auto Loader, checkpoints, Delta MERGE and watermarks can help implement incremental processing. The implementation must also handle late-arriving records and updates correctly.
9.4 SQL Warehouse cost optimization
- Select an appropriate warehouse size.
- Use auto-stop or serverless where appropriate.
- Avoid unnecessarily frequent dashboard refreshes.
- Optimize Gold tables and SQL queries.
- Monitor query duration and concurrency.
- Avoid running expensive exploratory queries repeatedly.
9.5 Production best practices
1. Use separate development, UAT and production environments.
2. Store source code and deployment configuration in Git.
3. Keep secrets out of code and Git history.
4. Use service principals for automated deployments.
5. Use least-privilege permissions.
6. Implement unit, integration and data-quality tests.
7. Use retries for transient failures and design idempotent jobs.
8. Monitor freshness, failure rates, runtime and compute cost.
9. Maintain rollback and recovery procedures.
10. Review changes before production deployment.
Databricks recommends version-controlled project files, environment isolation and staged testing. 
Microsoft Learn+1

10. Terraform vs Ansible vs Databricks Asset Bundles
These tools are related to DevOps but solve different problems.
Tool	Primary purpose	Example
Databricks Asset Bundles	Package and deploy Databricks workloads	Deploy a Lakeflow Job
Terraform	Provision and manage infrastructure	Create workspace and cloud resources
Ansible	Automate system configuration and operational tasks	Configure servers and install dependencies
GitHub Actions	Automate CI/CD workflows	Run tests and deploy a bundle
Jenkins	CI/CD automation server	Execute enterprise deployment pipelines

Interview answer: I would use Terraform for infrastructure provisioning, Asset Bundles for Databricks job and pipeline deployment, and GitHub Actions or Jenkins to automate testing and deployment.

You do not need to use all of them in every project.
11. Team responsibilities in a production Data Engineering team
Role	Responsibility
Data Engineer	Develop PySpark, SQL, ETL and data-quality logic
DevOps Engineer	Build and maintain CI/CD automation
Platform Engineer	Manage Databricks platform, compute, networking and governance
Data Architect	Design data models and platform architecture
QA / Test Engineer	Validate functionality, regression and integration tests
Data Analyst	Consume Gold data and develop business reporting

In a small team, one person may perform multiple responsibilities.
12. Interview questions and answers
Q1. What are Databricks Asset Bundles?

Declarative Automation Bundles, formerly called Databricks Asset Bundles, package source files and Databricks resources such as jobs and pipelines into a version-controlled deployable project.

Q2. Why do we use databricks.yml?

It is the main configuration file. It defines bundle metadata, resource includes, variables and deployment targets.

Q3. What is the difference between bundle validate and bundle deploy?

validate checks bundle configuration. deploy publishes the bundle resources and files to the target workspace.

Q4. How do you deploy the same job to dev, UAT and prod?

I define environment-specific targets in databricks.yml, configure the required variables and deploy using:
databricks bundle deploy -t dev
databricks bundle deploy -t uat
databricks bundle deploy -t prod


Q5. What is the Databricks CLI?

It is a command-line tool that allows users and automation systems to interact with Databricks workspaces and accounts.

Q6. How do you authenticate the CLI?

For local development, I use OAuth user login. For automated deployments, I use a service principal with OAuth M2M or workload identity federation.

Q7. What is CI?

Continuous Integration automatically builds and tests code changes so problems are identified before they are merged.

Q8. What is the difference between Continuous Delivery and Continuous Deployment?

Continuous Delivery keeps tested changes ready for production, usually with an approval step. Continuous Deployment automatically deploys changes after they pass the required checks.

Q9. How do you test PySpark transformations?

I separate transformation functions into Python modules and write pytest tests using small Spark DataFrames. I validate null handling, duplicate removal, calculations and business rules.

Q10. How do you reduce Databricks compute costs?

I use appropriate compute sizing, autoscaling, auto termination, incremental processing, optimized queries and monitoring to avoid unnecessary compute consumption.

Q11. Why use GitHub Actions?

It automates workflows such as installing dependencies, running tests, validating bundles and deploying code after approved changes.

Q12. What is the purpose of Terraform?

Terraform manages infrastructure as code. It can provision and maintain cloud and Databricks infrastructure.

Q13. What is idempotency in deployment and ETL?

An idempotent operation can be safely repeated without creating unintended duplicate or inconsistent results. It matters for retries and deployment recovery.

Q14. How would you deploy an E-Commerce Lakehouse job?

I would store the source code and job configuration in Git, write unit tests, validate the bundle, deploy to development, run integration tests in UAT and promote the approved configuration to production.

Q15. How would you roll back a failed production deployment?

I would identify the failed release, restore the previous known-good code and resource configuration through the deployment process, and validate the affected job and data. For data corruption, I would also use the appropriate Delta recovery or backfill strategy. Reverting Git alone does not automatically undo already-written data.